# Legal Guard NLP - Full Pipeline on Google Colab (GPU Accelerated)

This notebook runs the complete 5-stage Legal Guard pipeline:
1. Train 3 Model Families (TF-IDF, Legal-BERT, DeBERTa-v3) across 3 random seeds
2. Run Comparative Analysis (McNemar test, confusion matrices, error analysis)
3. Tune Two-Stage Risk Gating Filter (Recall floor = 0.70)
4. Evaluate Whole-Document Summarizer (Extractive, Abstractive, Hybrid)
5. Run full Pytest test suite

> **Tip:** Enable GPU before running: Go to **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.

## 1. Verify GPU Setup

In [ ]:
!nvidia-smi
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 2. Clone Repository or Mount Google Drive

In [ ]:
# Option A: Clone from GitHub
# !git clone https://github.com/Vedanti1703/legal-guard.git
# %cd legal-guard

# Option B: If uploaded to Google Drive as zip or folder:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/nlp

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q flask-cors

## 4. Run Full Pipeline via Bash Script

In [ ]:
# Give execution permission and run the automated pipeline
!chmod +x run_all.sh
!bash run_all.sh

## 5. Or Run Each Stage Individually

In [ ]:
# Stage 1: Train all models across 3 seeds (takes ~5-10 mins total on T4 GPU)
!python src/training/train_all_models.py --models tfidf,legalbert,deberta --seeds 42,7,2024 --epochs 2 --batch_size 32 --max_len 128

In [ ]:
# Stage 2: Comparative Analysis & Figures
!python src/evaluation/compare_models.py

In [ ]:
# Stage 3: Two-Stage Risk Gating Tuning
!python src/evaluation/tune_risk_filter.py --recall_floor 0.70

In [ ]:
# Stage 4: Summarizer Evaluation (ROUGE-1/2/L)
!python src/evaluation/evaluate_summarizer.py

In [ ]:
# Stage 5: Pytest Verification
!pytest tests/ -v

## 6. (Optional) Run Web UI via Public Tunnel (localtunnel or ngrok)

In [ ]:
# Install localtunnel to access the Flask web UI from Colab
!npm install -g localtunnel
import subprocess
import time

# Start Flask server in background
subprocess.Popen(["python", "server.py"])
time.sleep(3)

# Expose port 5000
!npx localtunnel --port 5000